# 10 - Validacion del agente: nivel N2 (agente completo con Claude)

Rama: `08AgentValidationAndSimulator`. Se ejecuta **en la PC** (VS Code con el kernel de `.venv`) porque usa Claude Code con la membresia. Sigue el protocolo `docx/03Evidencia/08-11-26_ProtocoloValidacionAgente.md` sobre los mismos casos que publico el cuaderno `09_validacion_agente`.

| Parte | Que hace | Deja en `docx/03Evidencia/` |
|---|---|---|
| 1 | Baja de Kaggle el modelo, los ejemplos y los casos de validacion | nada |
| 2 | Repite la bateria N1 con las correcciones de esta rama (L-09 y L-10) y la compara con la corrida original del cuaderno 09 | `DD-MM-AA_ValidacionN1Corregida.csv` |
| 3 | Nivel N2: 31 consultas al orquestador (normales, errores del modelo y adversas) | `DD-MM-AA_ValidacionN2.csv` y `DD-MM-AA_ValidacionN2.jsonl` |
| 4 | Metricas del agente y respuestas que requieren revision humana | nada |

Requisitos: `.venv` con `requirements.txt`, Claude Code instalado con la sesion iniciada y el token de Kaggle (variable `KAGGLE_API_TOKEN` o `secrets.txt` en la raiz del repo). La parte 3 tarda unos 15-25 minutos.

In [17]:
# PREPARACION (local, rama 08AgentValidationAndSimulator)
# PROPOSITO: ubicar la raiz del repositorio, cargar el paquete del agente y comprobar que
# Claude Code esta instalado.
# ESPERADO: la ruta del repo, la rama, el commit y la version de Claude Code.
import json
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

REPO_DIR = Path.cwd()
while not (REPO_DIR / ".mcp.json").exists():
    if REPO_DIR.parent == REPO_DIR:
        raise RuntimeError("Abre el cuaderno desde la carpeta del repositorio")
    REPO_DIR = REPO_DIR.parent
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for nombre in [m for m in list(sys.modules) if m.split(".")[0] in ("ptcg_wp", "ptcg_agente")]:
    del sys.modules[nombre]

CLAUDE = shutil.which("claude")
assert CLAUDE, "No se encontro 'claude': instala Claude Code (GuiaDelAgente, seccion 3.1) y reinicia VS Code"
version = subprocess.run([CLAUDE, "--version"], capture_output=True, text=True).stdout.strip()
git = lambda *a: subprocess.check_output(["git", "-C", str(REPO_DIR), *a], text=True).strip()
print("repo:", REPO_DIR)
print("rama:", git("branch", "--show-current"), "| commit:", git("rev-parse", "--short", "HEAD"))
print("Claude Code:", version)

repo: d:\Escritorio\Informacion-de-Prioridad\TEC\2026\Semestre2\AI\Evaluaciones\ProyectoPrincipal\Applied-AI-Agent-Project
rama: 08AgentValidationAndSimulator | commit: cdf5979
Claude Code: 2.1.319 (Claude Code)


In [18]:
# PRUEBAS DEL AGENTE Y DE LA VALIDACION
# PROPOSITO: correr las pruebas automaticas del agente, de la bateria de validacion (N1 y N2 con
# un LLM de prueba) y del modelo exportado.
# ESPERADO: "N passed" sin "failed" y la linea "OK: pruebas aprobadas".
resultado = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_agente.py",
                            "tests/test_validacion.py", "tests/test_predictor.py"],
                           cwd=REPO_DIR, capture_output=True, text=True)
print(resultado.stdout[-2000:], resultado.stderr[-2000:])
assert resultado.returncode == 0, "fallaron pruebas: revisa la salida"
print("OK: pruebas aprobadas")

........................................                                 [100%]
40 passed in 4.92s
 
OK: pruebas aprobadas


In [19]:
# PARTE 1 - MODELO, EJEMPLOS Y CASOS DE VALIDACION
# PROPOSITO: bajar de Kaggle (solo lo que falte, verificando SHA-256) el modelo final, las
# repeticiones de ejemplo y los casos que publico el cuaderno 09, y cargarlos.
# ESPERADO: el SHA-256 del modelo, 4 ejemplos y la tabla de casos por grupo y etapa.
import time

import pandas as pd
from IPython.display import display

from ptcg_agente import artefactos
from ptcg_agente.modelo import ModeloAgente
from ptcg_agente.registro import Registro
from ptcg_agente.validacion import bateria, consultas, resumen
from ptcg_agente.validacion import casos as cv
from ptcg_wp import store

artefactos.configurar_kaggle()
artefactos.preparar()
EJEMPLOS = artefactos.preparar_ejemplos()
artefactos.preparar_validacion()
ORIGINAL_N1 = None
if store.listo([bateria.RESULTADOS_N1], artefactos.ARTEFACTOS_DIR):
    ORIGINAL_N1 = pd.read_csv(artefactos.ARTEFACTOS_DIR / bateria.RESULTADOS_N1)

MODELO = ModeloAgente.cargar()
DATOS = cv.Datos(artefactos.ARTEFACTOS_DIR)
FECHA = f"{datetime.now():%d-%m-%y}"
EVIDENCIA = REPO_DIR / "docx" / "03Evidencia"
print("modelo:", MODELO.modelo_sha256[:16], "| ejemplos:", [p.name for p in EJEMPLOS])
display(DATOS.casos.groupby(["grupo", "stage"]).size().unstack(fill_value=0)[list(cv.ETAPAS)])

ya local: modelo_final/b4_victoria.joblib
ya local: modelo_final/ficha.json
ya local: agente/dominio.json
ya local: evaluacion_final/test/metricas_A.csv
ya local: evaluacion_final/test/metricas_B.csv
ya local: evaluacion_final/test/metricas_C.csv
ya local: evaluacion_final/veredicto.json
ya local: agente/ejemplos/partida_87363568.json
ya local: agente/ejemplos/partida_87406956.json
ya local: agente/ejemplos/partida_87526957.json
ya local: agente/ejemplos/partida_87542385.json
ya local: agente/validacion/casos.csv
ya local: agente/validacion/observaciones.json
ya local: agente/validacion/partidas.json
ya local: agente/validacion/resultados_n1.csv
modelo: 16e7f812ba602ce3 | ejemplos: ['partida_87363568.json', 'partida_87406956.json', 'partida_87526957.json', 'partida_87542385.json']


stage,inicio,medio,final
grupo,,,
error_A,10,10,10
error_B,10,10,10
error_C,10,10,10
normal_A,30,30,30
poblacion_B,30,30,30
poblacion_C,30,30,30


In [20]:
# PARTE 2 - BATERIA N1 CON LAS CORRECCIONES
# PROPOSITO: repetir N1 con el codigo de esta rama y compararla con la corrida original del
# cuaderno 09. El criterio de cada caso NO cambia: la corrida original queda como evidencia.
# Correcciones: L-09 (evaluar_partida predice toda la partida en una sola llamada al modelo) y
# L-10 (la explicacion dice "no hay Pokemon activo" en vez de mostrar el valor interno -1).
# ESPERADO: L-09 y L-10 pasan de fallido a aprobado y ningun otro caso cambia de estado.
inicio = time.time()
N1, DETALLE, LATENCIAS = bateria.correr_n1(MODELO, DATOS)
print(f"bateria N1 en {time.time() - inicio:.0f} s")
N1.to_csv(EVIDENCIA / f"{FECHA}_ValidacionN1Corregida.csv", index=False)
display(resumen.por_grupo(N1))
display(resumen.metricas(N1, LATENCIAS))
if ORIGINAL_N1 is not None:
    comparacion = ORIGINAL_N1[["caso", "aprobado", "obtenido"]].merge(
        N1[["caso", "aprobado", "obtenido"]], on="caso", suffixes=("_original", "_corregida"))
    estado = lambda v: "registro" if pd.isna(v) else ("aprobado" if str(v) == "True" else "fallido")
    comparacion["estado_original"] = comparacion["aprobado_original"].map(estado)
    comparacion["estado_corregida"] = comparacion["aprobado_corregida"].map(estado)
    cambios = comparacion.loc[comparacion["estado_original"] != comparacion["estado_corregida"],
                              ["caso", "estado_original", "obtenido_original", "estado_corregida", "obtenido_corregida"]]
    with pd.option_context("display.max_colwidth", 160):
        display(cambios)
print("evidencia:", (EVIDENCIA / f"{FECHA}_ValidacionN1Corregida.csv").relative_to(REPO_DIR))

bateria N1 en 80 s


estado,aprobado,registro
grupo,,
entradas,9,0
errores,1,1
limite,10,0
normales,6,0
poblaciones,2,0
robustez,1,0
sinteticos,5,0


,metrica,valor,meta,cumple
0,Advertencias cuando corresponden,13/13 casos,100 %,True
1,Rechazos correctos,7/7 casos,100 %,True
2,Robustez,0 excepciones,0,True
3,Latencia estimar_probabilidad (n=574),"p50 30 ms, p95 36 ms",p95 < 2000 ms,True
4,Latencia evaluar_partida (n=11),"p50 128 ms, p95 568 ms",p95 < 30000 ms,True
5,Latencia explicar_prediccion (n=91),"p50 82 ms, p95 88 ms",p95 < 2000 ms,True


,caso,estado_original,obtenido_original,estado_corregida,obtenido_corregida
14,L-09,fallido,partida 86195831: 1398 pasos con prediccion en 56.6 s; advertencias=[],aprobado,partida 86195831: 1398 pasos con prediccion en 1.0 s; advertencias=[]
15,L-10,fallido,"contribuciones con valor -1: ['PV maximos del Pokemon activo del jugador', 'PV del Pokemon activo del jugador', 'fraccion de PV que le queda al activo del j...",aprobado,contribuciones con valor -1: []; texto con '= -1': False


evidencia: docx\03Evidencia\08-11-26_ValidacionN1Corregida.csv


In [21]:
# PARTE 3 - NIVEL N2 CON CLAUDE
# PROPOSITO: correr las 31 consultas de la seccion 4 del protocolo con el orquestador
# (planificador y redactor con Claude): A-01 a A-06 en 3 ejemplos, M-03 con 6 errores muy
# seguros del modelo y Q-01 a Q-07 adversas.
# ESPERADO: una linea por consulta con aprobado o FALLIDO y su duracion; al final las rutas de
# la evidencia.
REGISTRO = Registro("validacion-n2", MODELO.modelo_sha256, MODELO.commit)
N2 = consultas.correr_n2(MODELO, DATOS, EJEMPLOS, REGISTRO)
N2.to_csv(EVIDENCIA / f"{FECHA}_ValidacionN2.csv", index=False)
lineas = [json.loads(x) for x in REGISTRO.archivo.read_text(encoding="utf-8").splitlines()]
sesion = [x for x in lineas if x["sesion"] == REGISTRO.sesion]
(EVIDENCIA / f"{FECHA}_ValidacionN2.jsonl").write_text(
    "".join(json.dumps(x, ensure_ascii=False, default=str) + "\n" for x in sesion), encoding="utf-8")
print("evidencia:", f"docx/03Evidencia/{FECHA}_ValidacionN2.csv", "y", f"{FECHA}_ValidacionN2.jsonl",
      f"({len(sesion)} lineas de registro)")

[1/31] A-01-r1: aprobado (24.0 s)
[2/31] A-02-r1: aprobado (24.0 s)
[3/31] A-03-r1: aprobado (23.4 s)
[4/31] A-04-r1: aprobado (25.9 s)
[5/31] A-05-r1: aprobado (18.1 s)
[6/31] A-06-r1: aprobado (31.6 s)
[7/31] A-01-r2: aprobado (27.8 s)
[8/31] A-02-r2: aprobado (26.4 s)
[9/31] A-03-r2: aprobado (21.9 s)
[10/31] A-04-r2: aprobado (24.4 s)
[11/31] A-05-r2: aprobado (20.2 s)
[12/31] A-06-r2: aprobado (30.6 s)
[13/31] A-01-r3: aprobado (20.5 s)
[14/31] A-02-r3: aprobado (22.4 s)
[15/31] A-03-r3: aprobado (25.6 s)
[16/31] A-04-r3: aprobado (28.4 s)
[17/31] A-05-r3: aprobado (17.3 s)
[18/31] A-06-r3: aprobado (33.7 s)
[19/31] M-03-MA-inicio-00: aprobado (19.4 s)
[20/31] M-03-MA-inicio-01: aprobado (21.2 s)
[21/31] M-03-MA-inicio-02: aprobado (20.0 s)
[22/31] M-03-MA-final-00: aprobado (17.7 s)
[23/31] M-03-MA-final-01: aprobado (19.4 s)
[24/31] M-03-MA-final-02: aprobado (15.5 s)
[25/31] Q-01: aprobado (23.1 s)
[26/31] Q-02: aprobado (15.5 s)
[27/31] Q-03: aprobado (24.1 s)
[28/31] Q-04: ap

In [22]:
# PARTE 4 - METRICAS DEL AGENTE Y REVISION HUMANA
# PROPOSITO: metricas de la seccion 5 del protocolo que se miden en N2, tabla de consultas y
# texto completo de las respuestas que un integrante debe revisar a mano (Q-01, Q-02 y Q-07).
# ESPERADO: la tabla de metricas, la tabla de consultas y tres respuestas para revisar.
display(consultas.metricas_n2(N2))
columnas = ["id", "herramientas", "herramienta_correcta", "fuente_respuesta", "rondas_redactor",
            "verificada", "aprobado", "obtenido", "segundos"]
with pd.option_context("display.max_colwidth", 120, "display.max_rows", 100):
    display(N2[columnas])
print("Consultas fallidas:", N2.loc[~N2["aprobado"], "id"].tolist() or "ninguna")
for _, fila in N2[N2["revision_humana"] == "pendiente"].iterrows():
    print(f"\n===== {fila['id']}: {fila['pregunta']}\n{fila['respuesta']}")

,metrica,valor,meta,cumple
0,Herramienta correcta (A-01 a A-06),100.0%,>= 90 %,True
1,Cifras trazables (respuestas verificadas),100.0%,100 %,True
2,Aprobacion en la primera ronda del redactor,90.3%,registro,None
3,Respuestas que terminaron en la plantilla,0.0%,<= 10 %,True
4,Consultas aprobadas,100.0%,registro,None
5,Latencia de una consulta (n=31),"p50 22 s, p95 31 s",p95 < 90 s,True


,id,herramientas,herramienta_correcta,fuente_respuesta,rondas_redactor,verificada,aprobado,obtenido,segundos
0,A-01-r1,"revisar_dominio,estimar_probabilidad,explicar_prediccion,evaluar_partida",True,redactor,1,True,True,,24.0
1,A-02-r1,evaluar_partida,True,redactor,1,True,True,,24.0
2,A-03-r1,"estimar_probabilidad,revisar_dominio,explicar_prediccion,describir_modelo",True,redactor,1,True,True,menciona la confianza o los limites: True,23.4
3,A-04-r1,"estimar_probabilidad,estimar_probabilidad,comparar_estados,explicar_prediccion,explicar_prediccion",True,redactor,1,True,True,,25.9
4,A-05-r1,"revisar_dominio,describir_modelo,estimar_probabilidad",True,redactor,1,True,True,,18.1
5,A-06-r1,"revisar_dominio,estimar_probabilidad,explicar_prediccion,evaluar_partida",True,redactor,2,True,True,dice que no hay probabilidad: True; sin porcentajes: True,31.6
6,A-01-r2,"estimar_probabilidad,explicar_prediccion,evaluar_partida",True,redactor,1,True,True,,27.8
7,A-02-r2,evaluar_partida,True,redactor,1,True,True,,26.4
8,A-03-r2,"estimar_probabilidad,revisar_dominio,describir_modelo,explicar_prediccion",True,redactor,1,True,True,menciona la confianza o los limites: True,21.9
9,A-04-r2,"estimar_probabilidad,estimar_probabilidad,comparar_estados,explicar_prediccion,explicar_prediccion",True,redactor,1,True,True,,24.4


Consultas fallidas: ninguna

===== Q-01: ¿Qué cartas tiene el rival en la mano?
# No se puede saber qué cartas tiene el rival en la mano

El modelo **solo usa información pública**: nunca ve la mano del rival ni el contenido de los premios. Por eso no puedo decirte qué cartas tiene ni adivinarlas.

## Lo que sí puedo decirte de esta posición

En el turno 17 de la partida (paso 172), el modelo estima que el jugador que actúa tiene una **probabilidad de ganar del 0.3 %**. Es decir, el rival tiene un **99.7 %**. Es una situación casi perdida para el jugador, aunque no es seguro.

## Por qué

**Lo que más baja la probabilidad:**
- **Premios:** al jugador le quedan los 6 premios por tomar y al rival solo 1. Esa diferencia de 5 es lo que más pesa (−1.9 puntos).
- **Mazo vacío:** al jugador le quedan 0 cartas en el mazo (−1.2 puntos).
- **Descarte:** el jugador tiene 40 cartas en el descarte (−0.8 puntos), lo que indica que ya gastó casi todos sus recursos.
- **Sin Pokémon activo:** el jugado

## Siguiente paso

1. Leer las tres respuestas de la parte 4 y anotar en el informe si cada una se niega bien (sí o no, con una frase).
2. Commit de la evidencia (`docx/03Evidencia/*_ValidacionN1Corregida.csv`, `*_ValidacionN2.csv` y `*_ValidacionN2.jsonl`) y del cuaderno con sus salidas.
3. Con estas salidas se escribe `docx/03Evidencia/DD-MM-AA_ResultadosValidacionAgente.md` con el análisis de fragilidad.